In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segmentation-models-pytorch==0.3.4"], check=True)
print("installed smp 0.3.4")

installed smp 0.3.4


In [2]:
import os, glob, random
import numpy as np
import torch
import torch.nn as nn
import cv2
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from PIL import Image
from tqdm.auto import tqdm
import segmentation_models_pytorch as smp
import pandas as pd

In [3]:
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

ROOT     = "/kaggle/input/competitions/2026task-4/segmentation"
IMG_DIR  = os.path.join(ROOT, "train", "image")
LBL_DIR  = os.path.join(ROOT, "train", "label")
TEST_DIR = os.path.join(ROOT, "test", "image")
SCRIPT   = os.path.join(ROOT, "segmentation_to_csv.py")
WORK     = "/kaggle/working"
PRED_DIR = os.path.join(WORK, "image")

SIZE   = 768
BATCH  = 1
EPOCHS = 60
LR     = 5e-4
THRESH = 0.6
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [4]:
def list_names(d):
    return sorted(os.path.splitext(os.path.basename(p))[0]
                  for p in glob.glob(os.path.join(d, "*")))

def read_img(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    g = img[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = img.copy()
    img[:, :, 1] = clahe.apply(g)
    return img

def read_lbl(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return (m > 0).astype(np.uint8)

train_tf = A.Compose([
    A.Resize(SIZE, SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.95,1.05), translate_percent=(-0.03,0.03),
             rotate=(-15,15), p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.1),
])
val_tf = A.Compose([A.Resize(SIZE, SIZE)])

class SegDataset(Dataset):
    def __init__(self, names, tf):
        self.names, self.tf = names, tf
    def __len__(self): return len(self.names)
    def __getitem__(self, i):
        n = self.names[i]
        ip = glob.glob(os.path.join(IMG_DIR, n + ".*"))[0]
        lp = glob.glob(os.path.join(LBL_DIR, n + ".*"))[0]
        out = self.tf(image=read_img(ip), mask=read_lbl(lp))
        img = np.transpose(out["image"].astype(np.float32)/255.0, (2,0,1))
        lbl = out["mask"].astype(np.float32)[None]
        return torch.from_numpy(img), torch.from_numpy(lbl)

names = list_names(IMG_DIR)
print("train:", len(names))

train: 20


In [5]:
def build_model():
    return smp.Unet(encoder_name="resnet18", encoder_weights="imagenet",
                    in_channels=3, classes=1).to(DEVICE)

bce = nn.BCEWithLogitsLoss()

def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    num = 2*(p*target).sum(dim=(2,3)) + eps
    den = p.sum(dim=(2,3)) + target.sum(dim=(2,3)) + eps
    return 1 - (num/den).mean()

def tversky_loss(logits, target, alpha=0.3, beta=0.8, eps=1e-6):
    p = torch.sigmoid(logits)
    tp = (p * target).sum(dim=(2,3))
    fp = (p * (1 - target)).sum(dim=(2,3))
    fn = ((1 - p) * target).sum(dim=(2,3))
    return 1 - ((tp + eps) / (tp + alpha*fp + beta*fn + eps)).mean()

def dice_score(logits, target, eps=1e-6):
    p = (torch.sigmoid(logits) > THRESH).float()
    num = 2*(p*target).sum(dim=(2,3)) + eps
    den = p.sum(dim=(2,3)) + target.sum(dim=(2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def predict_tta(model, x):
    model.eval()
    ps = []
    for mode in [0, 1, 2]:
        xx = x
        if mode == 1: xx = torch.flip(x, dims=[3])
        if mode == 2: xx = torch.flip(x, dims=[2])
        p = torch.sigmoid(model(xx))
        if mode == 1: p = torch.flip(p, dims=[3])
        if mode == 2: p = torch.flip(p, dims=[2])
        ps.append(p)
    return torch.stack(ps).mean(0)[0,0].cpu().numpy()

In [6]:
set_seed()
idx = np.arange(len(names))
rng = np.random.RandomState(SEED); rng.shuffle(idx)
folds = np.array_split(idx, 5)

all_probs = {n: np.zeros((SIZE, SIZE), np.float32)
             for n in list_names(TEST_DIR)}
fold_dices = []

for fi, val_idx in enumerate(folds):
    set_seed(SEED + fi)
    tr_idx = np.setdiff1d(np.arange(len(names)), val_idx)
    tr_dl = DataLoader(SegDataset([names[i] for i in tr_idx], train_tf),
                       batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
    va_dl = DataLoader(SegDataset([names[i] for i in val_idx], val_tf),
                       batch_size=1, shuffle=False, num_workers=2)

    m = build_model()
    opt = AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sch = CosineAnnealingLR(opt, T_max=EPOCHS)
    best, no_improve = -1, 0

    for ep in tqdm(range(EPOCHS), desc=f"fold {fi+1}/5", leave=False):
        m.train()
        for x, y in tr_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logit = m(x)
            loss = 0.3*bce(logit, y) + 0.7*tversky_loss(logit, y)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()

        m.eval(); ds = []
        with torch.no_grad():
            for x, y in va_dl:
                x, y = x.to(DEVICE), y.to(DEVICE)
                ds.append(dice_score(m(x), y))
        vd = float(np.mean(ds))
        if vd > best:
            best, no_improve = vd, 0
            torch.save(m.state_dict(), f"{WORK}/fold{fi}_best.pth")
        else:
            no_improve += 1
            if no_improve >= 15:
                break

    fold_dices.append(best)
    print(f"fold {fi+1}: Dice={best:.4f}")

    m.load_state_dict(torch.load(f"{WORK}/fold{fi}_best.pth"))
    m.eval()
    for n in all_probs:
        tp = glob.glob(os.path.join(TEST_DIR, n + ".*"))[0]
        img = cv2.resize(read_img(tp), (SIZE, SIZE), interpolation=cv2.INTER_LINEAR)
        x = torch.from_numpy(np.transpose(img.astype(np.float32)/255.0,(2,0,1)))[None].to(DEVICE)
        all_probs[n] += predict_tta(m, x) / len(folds)

print("各折 Dice:", [f"{d:.4f}" for d in fold_dices])
print("平均验证 Dice:", float(np.mean(fold_dices)))

Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 210MB/s]


fold 1/5:   0%|          | 0/60 [00:00<?, ?it/s]

fold 1: Dice=0.7693


fold 2/5:   0%|          | 0/60 [00:00<?, ?it/s]

fold 2: Dice=0.7613


... (2 outputs left out)

fold 4/5:   0%|          | 0/60 [00:00<?, ?it/s]

fold 4: Dice=0.8009


fold 5/5:   0%|          | 0/60 [00:00<?, ?it/s]

fold 5: Dice=0.7917
各折 Dice: ['0.7693', '0.7613', '0.8081', '0.8009', '0.7917']
平均验证 Dice: 0.7862580120563507


In [7]:
set_seed(123)
full_dl = DataLoader(SegDataset(names, train_tf), batch_size=BATCH,
                     shuffle=True, num_workers=2, drop_last=True)
m_full = build_model()
opt = AdamW(m_full.parameters(), lr=LR, weight_decay=1e-4)
sch = CosineAnnealingLR(opt, T_max=100)

for ep in tqdm(range(100), desc="full", ncols=100):
    m_full.train()
    for x, y in full_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logit = m_full(x)
        loss = 0.3*bce(logit, y) + 0.7*tversky_loss(logit, y)
        opt.zero_grad(); loss.backward(); opt.step()
    sch.step()

m_full.eval()
for n in all_probs:
    tp = glob.glob(os.path.join(TEST_DIR, n + ".*"))[0]
    img = cv2.resize(read_img(tp), (SIZE, SIZE), interpolation=cv2.INTER_LINEAR)
    x = torch.from_numpy(np.transpose(img.astype(np.float32)/255.0,(2,0,1)))[None].to(DEVICE)
    prob = predict_tta(m_full, x)
    all_probs[n] = all_probs[n] * (5/6) + prob * (1/6)

full:   0%|                                                                 | 0/100 [00:00<?, ?it/s]

In [8]:
import shutil
if os.path.exists(PRED_DIR): shutil.rmtree(PRED_DIR)
os.makedirs(PRED_DIR, exist_ok=True)

for n, prob in all_probs.items():
    msk = (prob > THRESH).astype(np.uint8)
    num, lab, stats_, _ = cv2.connectedComponentsWithStats(msk, 8)
    clean = np.zeros_like(msk)
    for i in range(1, num):
        if stats_[i, cv2.CC_STAT_AREA] >= 50:
            clean[lab == i] = 1
    Image.fromarray(clean * 255, mode="L").save(os.path.join(PRED_DIR, n + ".png"))
print("saved:", len(os.listdir(PRED_DIR)))

saved: 20


In [9]:
shutil.copy(SCRIPT, WORK)
os.chdir(WORK)
!python segmentation_to_csv.py

In [10]:
sample = pd.read_csv(os.path.join(ROOT, "sample_submission.csv"))
sub = pd.read_csv(os.path.join(WORK, "submission.csv"))
sub = sample[["Id"]].merge(sub, on="Id", how="left")
print("Ids:", sub["Id"].tolist(), "空值:", sub["Predicted"].isna().sum())
assert sub["Predicted"].notna().all()
sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
print("shape:", sub.shape)

Ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] 空值: 0
shape: (20, 2)


In [11]:
sub = pd.read_csv(os.path.join(WORK, "submission.csv"))
print("shape:", sub.shape)
print("Ids:", sub["Id"].tolist())
print("空值:", sub["Predicted"].isna().sum())

def decode(rle, h=512, w=512):
    a = np.zeros(h*w, np.uint8)
    if isinstance(rle, str) and rle.strip():
        nums = list(map(int, rle.split()))
        for s, l in zip(nums[0::2], nums[1::2]):
            a[s-1:s-1+l] = 1
    return a.reshape(h, w)

m = decode(sub.iloc[0]["Predicted"])
print("第1行前景像素:", int(m.sum()))
print(sub.head())

shape: (20, 2)
Ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
空值: 0
第1行前景像素: 33251
   Id                                          Predicted
0   1  13560 4 14071 6 14583 7 15095 7 15608 6 16121 ...
1   2  16097 3 16608 4 17120 4 17632 4 18144 4 18656 ...
2   3  15111 3 15622 4 16093 5 16134 4 16154 3 16605 ...
3   4  16130 4 16641 5 16674 6 17150 8 17186 6 17662 ...
4   5  15618 6 16128 8 16639 9 17150 9 17662 8 18172 ...
